In [1]:
!git clone https://github.com/opendatalab/MinerU-HTML
%cd MinerU-HTML
!pip install .
!pip install vllm  # Fondamentale per il backend

Cloning into 'MinerU-HTML'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 89 (delta 19), reused 14 (delta 14), pack-reused 59 (from 1)
Receiving objects: 100% (89/89), 103.15 KiB | 844.00 KiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/MinerU-HTML
Processing /content/MinerU-HTML
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install huggingface_hub

In [2]:
from huggingface_hub import snapshot_download

# Scarica il modello (sono circa 8-10GB, Colab li scarica velocemente)
model_path = snapshot_download(repo_id="opendatalab/MinerU-HTML")
print(f"Modello scaricato in: {model_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Modello scaricato in: /root/.cache/huggingface/hub/models--opendatalab--MinerU-HTML/snapshots/2498a57895e29ccf9c1d5dd60e5f68f18c1406c6


In [ ]:
from dripper.api import Dripper
import os

# Inizializzazione
dripper = Dripper(
    config={
        'model_path': model_path,
        'use_fall_back': True,
        'inference_backend': "vllm",
        "model_init_kwargs": {
          'tensor_parallel_size': 1,
          'gpu_memory_utilization': 0.8, # un po' di pietà alla T4
        },
    }
)

# Test su un frammento sporco
html_test = """
<html>
  <body>
    <nav>Menu inutile, pubblicità, banner cookie</nav>
    <div class="main-content">
      <h1>Sciopero Nazionale Trasporti 24h</h1>
      <p>Il sindacato USB comunica che il giorno 15 marzo...</p>
    </div>
    <aside>Articoli correlati e spam</aside>
  </body>
</html>
"""

result = dripper.process(html_test)
print("--- CONTENUTO PULITO ---")
print(result[0].main_html)

INFO 03-09 18:14:24 [utils.py:253] non-default args: {'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': '/root/.cache/huggingface/hub/models--opendatalab--MinerU-HTML/snapshots/2498a57895e29ccf9c1d5dd60e5f68f18c1406c6'}
INFO 03-09 18:14:47 [model.py:631] Resolved architecture: Qwen3ForCausalLM
WARNING 03-09 18:14:47 [model.py:1921] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 03-09 18:14:47 [model.py:1971] Casting torch.bfloat16 to torch.float16.
INFO 03-09 18:14:47 [model.py:1745] Using max model len 40960
INFO 03-09 18:14:47 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 03-09 18:14:47 [system_utils.py:103] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA i

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

--- CONTENUTO PULITO ---
<html><body>
<div class="main-content">
<h1 _item_id="1">Sciopero Nazionale Trasporti 24h</h1>
<p _item_id="2">Il sindacato USB comunica che il giorno 15 marzo...</p>
</div>
</body></html>


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json

# from google.colab import drive
# drive.mount('/content/drive')

def clean_my_dataset(input_folder, output_folder):
    files = [f for f in os.listdir(input_folder) if f.endswith('.html')]

    for file_name in files:
        with open(f"{input_folder}/{file_name}", "r") as f:
            raw_html = f.read()

        print(f"Sto processando: {file_name}")

        # Estrazione con MinerU
        cleaned_result = dripper.process(raw_html)
        main_content = cleaned_result[0].main_html

        # Salva il risultato (Markdown o HTML pulito)
        output_name = file_name.replace(".html", "_clean.html")
        with open(f"{output_folder}/{output_name}", "w") as f:
            f.write(main_content)
        print(f"Processato: {file_name}")


In [ ]:
# clean_my_dataset('/content/drive/MyDrive/tesi/raw-data/trenitalia', '/content/drive/MyDrive/tesi/cleaned-data/trenitalia')

In [6]:
clean_my_dataset('/content/drive/MyDrive/tesi/raw-data/atac', '/content/drive/MyDrive/tesi/cleaned-data/atac')

Sto processando: sciopero_trasporto_pubblico_adesione_personale_atac_209.html


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processato: sciopero_trasporto_pubblico_adesione_personale_atac_209.html
Sto processando: sciopero_nazionale_adesione_personale_atac_al_237.html


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processato: sciopero_trasporto_pubblico_adesione_personale_atac_144.html
Sto processando: sciopero_trasporto_pubblico_ha_aderito_246_personale_atac.html


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processato: sciopero_trasporto_pubblico_ha_aderito_246_personale_atac.html
